# Handling Metric Collapse
The updates made to the Bag of Tricks (BoT) architecture in model.txt are exactly what is needed to address the "Metric Collapse" and improve the Triplet branch performance

## Step A
1. A `Dense layer` actnd as `shared branch` is added after Flatten the GeMPooling. The output 
of shared branch act as input for Triplet Loss Branch (L2Normalizer) and BNeck Branch (classification)

2. For the BNNeck to work correctly, the classification head must use the output of that new Dense layer. `This allows the bottleneck to normalize the lower-dimensional embedding rather than the raw pooling output`

## Step B: 
`Re-balance the Loss WeightsYour id_loss ($0.59$)` is currently double your gem_loss ($0.29$). The optimizer is naturally prioritizing the classification. You must force it to care about the distances.


# What success looks like
After making these changes and re-training:

1. Hard Positive Distance: Should drop significantly compared to the negative.
2. Hard Negative Distance: Should be pushed toward 0.3 - 0.5.
3. EMNIST Test Loss: Your gem_loss should start dropping below 0.2.
4. OOS Accuracy: Once the "buckets" for each digit are spread out on the hypersphere, the Font-based digits will have "room" to land in the correct class, pushing your 60% accuracy toward 80%+.

# Imports,And Loading Dataset

In [ ]:
import os
import time
import numpy as np
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
import tensorflow.keras.backend as K
from sklearn.manifold import TSNE
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers, backend, losses
from tensorflow.keras.utils import plot_model # Added plot_model
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score, confusion_matrix,
                             classification_report, accuracy_score)
from sklearn.preprocessing import label_binarize

# Download latest version
DATA_DIR = "/kaggle/input/emnist-digits-balanced"

# Global access and constants
IMG_SIZE = (28, 28)
BATCH_SIZE = 128
SEED = 42
shift_val = 1.0 / 28

# Define parameters based on the paper
TARGET_LR = 3.5e-4
WARMUP_EPOCHS = 10


def get_keras_dataset():
    print("--- Loading EMNIST Digits from local PNG folders ---")

    # Global access and constants
    global IMG_SIZE
    global BATCH_SIZE
    global SEED
    global shift_val

    # 1. Load TRAIN dataset
    train_ds = tf.keras.utils.image_dataset_from_directory(
        f"{DATA_DIR}/train",
        validation_split=0.2,
        subset="training",
        seed=SEED,
        color_mode="grayscale",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        f"{DATA_DIR}/train",
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        color_mode="grayscale",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )

    # 2. Load TEST dataset
    test_ds = tf.keras.utils.image_dataset_from_directory(
        directory=f"{DATA_DIR}/test",
        labels="inferred",
        label_mode='categorical',
        color_mode="grayscale",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    class_names = train_ds.class_names
    print("Class names:", class_names)

    # 4. Normalization layer (NO augmentation)
    norm_layer = layers.Normalization(axis=None)
    # Adapt on a mapping to ensure the stats are based on [0, 1] range
    norm_layer.adapt(
        train_ds.map(lambda x, y: x / 255.0).take(500)
    )

    # Fixed 1-pixel translation layer
    translation_layer = layers.RandomTranslation(
        height_factor=(shift_val, shift_val), # Fixed at +1 pixel vertically
        width_factor=(shift_val, shift_val),  # Fixed at +1 pixel horizontally
        fill_mode='constant',
        fill_value=0.0,
        seed=SEED
    )

    preprocessing_model = models.Sequential([
        layers.Rescaling(1./255),
        norm_layer,
        layers.RandomRotation(0.3),
        translation_layer,
        layers.RandomZoom(0.02)
    ], name="preprocessing_head")

    # 5. Final pipeline
    def finalize(ds, shuffle=False):
        if shuffle:
            ds = ds.shuffle(10000, seed=SEED)
        return ds.prefetch(tf.data.AUTOTUNE)

    train_ds = finalize(train_ds, shuffle=True)
    val_ds = finalize(val_ds)
    test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds, test_ds, preprocessing_model

# Build And Train Model

In [ ]:
# -------------------------------
# ᐧ GEM Pooling
# -------------------------------
@tf.keras.utils.register_keras_serializable(package="CustomLayers")
class GeMPooling2D(layers.Layer):
    def __init__(self, p=3.0, trainable=True, epsilon=1e-6, **kwargs):
        super(GeMPooling2D, self).__init__(**kwargs)
        self.p = p
        self.trainable = trainable
        self.epsilon = epsilon

    def build(self, input_shape):
        self.p_param = self.add_weight(
            name="p",
            shape=(1, 1, 1, input_shape[-1]),
            initializer=tf.keras.initializers.Constant(self.p),
            trainable=self.trainable,
            # Use a formal Keras constraint object
            constraint=tf.keras.constraints.MinMaxNorm(min_value=1.0, max_value=10.0, rate=1.0, axis=0),
            dtype=tf.float32
        )
        super(GeMPooling2D, self).build(input_shape)

    def call(self, x):
        # Ensure input is positive for pow operations
        x = tf.maximum(x, self.epsilon)
        pow_mean = tf.reduce_mean(tf.pow(x, self.p_param), axis=[1, 2], keepdims=True)
        gem_out = tf.pow(pow_mean, 1.0 / self.p_param)
        return tf.reshape(gem_out, [tf.shape(x)[0], tf.shape(x)[-1]])

    def get_config(self):
        config = super().get_config()
        config.update({
            "p": self.p,
            "trainable": self.trainable,
            "epsilon": self.epsilon
        })
        return config

    def compute_output_shape(self, input_shape):
        # GeM collapses spatial dimensions, leaving only (batch, channels)
        return (input_shape[0], input_shape[-1])


def triplet_semi_hard_loss(labels, embeddings, margin=0.3):
    """
    Custom implementation of Triplet Semi-Hard Loss.
    labels: [batch_size, 1] or [batch_size]
    embeddings: [batch_size, embedding_dim] (These are the GeM features)
    """
    # 1. Compute Pairwise Distance Matrix: dist(i, j) = ||e_i - e_j||^2
    # Using the formula: ||a-b||^2 = ||a||^2 - 2ab + ||b||^2
    squared_norm = tf.reduce_sum(tf.square(embeddings), axis=1, keepdims=True)
    distances = squared_norm - 2.0 * tf.matmul(embeddings, embeddings, transpose_b=True) + tf.transpose(squared_norm)
    distances = tf.maximum(distances, 0.0) # Avoid negative numbers due to float errors

    # 2. Build masks for Positive and Negative pairs
    labels = tf.reshape(labels, [-1, 1])
    mask_pos = tf.equal(labels, tf.transpose(labels))
    mask_pos = tf.cast(mask_pos, tf.float32)

    mask_neg = tf.logical_not(tf.equal(labels, tf.transpose(labels)))
    mask_neg = tf.cast(mask_neg, tf.float32)

    # 3. Calculate distances
    # For each anchor, find the distance to all positives
    anchor_positive_dist = tf.multiply(mask_pos, distances)
    # Get the hardest positive (max distance) for each anchor
    hardest_positive_dist = tf.reduce_max(anchor_positive_dist, axis=1)

    # For each anchor, find the semi-hard negatives
    # Semi-hard: d(a,p) < d(a,n) < d(a,p) + margin
    # To simplify for training, we often just use "Hard Negative Mining":
    # (Select the smallest distance where labels are different)
    max_dist = tf.reduce_max(distances)
    negative_dist = distances + max_dist * (1.0 - mask_neg) # Push positive distances to infinity
    hardest_negative_dist = tf.reduce_min(negative_dist, axis=1)

    # 4. Final Loss
    loss = tf.maximum(hardest_positive_dist - hardest_negative_dist + margin, 0.0)
    return tf.reduce_mean(loss)


# Wrap the custom function to match Keras Loss API
@tf.keras.utils.register_keras_serializable(package="CustomLayers")
class TripletSemiHardLoss(tf.keras.losses.Loss):
    def __init__(self, margin=0.3, name="triplet_loss", **kwargs):
        # **kwargs captures 'reduction' and other Keras-internal arguments
        super().__init__(name=name, **kwargs)
        self.margin = margin

    def call(self, y_true, y_pred):
        # y_true will be the labels (e.g., person IDs)
        # y_pred will be the 'gem_features'
        return triplet_semi_hard_loss(y_true, y_pred, margin=self.margin)

    def get_config(self):
        """
        Returns the config dictionary for the loss.
        This is required for saving and loading models.
        """
        config = super().get_config()
        config.update({
            "margin": self.margin
        })
        return config

@tf.keras.utils.register_keras_serializable(package="CustomLayers")
class L2NormalizeLayer(tf.keras.layers.Layer):
    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=1)


def mobilenet_v1_block(x, filters, stride=1):
  l2_reg = regularizers.l2(1e-4)

  # 1. Depthwise Convolution: Filters each input channel individually
  # Parameters for a standard Depthwise layer in MobileNetV1
  x = keras.layers.DepthwiseConv2D(
      kernel_size=(3, 3),             # Paper uses 3x3 for all Depthwise layers
      strides=(stride, stride),       # Use the 'stride' argument here
      padding="same",                 # Paper uses zero-padding to maintain/halve resolution
      depth_multiplier=1,             # Standard V1 setup
      activation=None,                # Activation (ReLU6) comes AFTER Batch Normalization
      use_bias=False,                 # Crucial: BN makes bias redundant
      depthwise_initializer="he_normal", # Optimized for ReLU/ReLU6
      depthwise_regularizer= l2_reg # Weight decay from the paper
  )(x) # Apply the layer to input 'x'
  x = layers.BatchNormalization()(x)
  x = layers.ReLU(6.0)(x)  # ReLU6 caps output at 6 for mobile efficiency

  # 2. Pointwise Convolution: 1x1 conv to combine channels
  x = layers.Conv2D(filters=filters, kernel_size=1, strides=1,
                  padding='same', use_bias=False,
                  kernel_regularizer=l2_reg,
                  kernel_initializer='he_normal')(x)
  x = layers.BatchNormalization()(x)
  x = layers.ReLU(6.0)(x)

  return x

def build_mobilenet_v1_bnneck_model(preprocessing_layer):
    inputs = layers.Input(shape=(None, None, 1))
    x = preprocessing_layer(inputs)

    # --- Feature Extraction Backbone ---
    # Using 3 blocks for EMNIST (input->14->14->7)
    x = mobilenet_v1_block(x, 8, stride=1)
    x = mobilenet_v1_block(x, 16, stride=2)
    x = mobilenet_v1_block(x, 32, stride=1)
    x = mobilenet_v1_block(x, 48, stride=2)
    x = mobilenet_v1_block(x, 64, stride=1)
    x = mobilenet_v1_block(x, 80, stride=1)
    
    # --- GeM Pooling ---
    # Output: (batch, 64)
    raw_gem_feat = GeMPooling2D(p=3.0, name='raw_gem_embedding_layer')(x)
    
    x = layers.Flatten()(raw_gem_feat)
    # This is the shared embedding that both branches should use
    shared_embeddings = layers.Dense(embedding_dim, name='feature_embedding')(x)

    # --- Branch 1: Metric Learning (Triplet Loss) ---
    # L2 Normalize for Triplet Loss (Required by triplet loss)
    # This ensures embeddings are on a unit hypersphere
    # This ensures d(A,P) and d(A,N) are on a fixed scale
    # Now uses the DENSE features, not the raw pooling
    global_feat = L2NormalizeLayer(name='gem_features')(shared_embeddings)

    # -----------------------------------------------------------
    # BNNeck Implementation
    # -----------------------------------------------------------
    # 1. 'global_feat' is used for Triplet Loss (Euclidean space)

    # 2. The BN layer "Neck"
    # We set center=True, scale=True as per the TMM 2020 paper

    # --- Branch 2: Classification (ID Loss) ---
    # Connect BNNeck to the shared_embeddings
    bn_feat = layers.BatchNormalization(name='bn_neck')(shared_embeddings)
    
    # 3. Classifier Head for ID Loss (Cosine space)
    # use_bias=False is crucial here
    probabilities = layers.Dense(
        10,
        use_bias=False,
        activation='softmax',
        name='id_loss_output')(bn_feat)

    # 4. Create Model with TWO outputs
    # This allows us to pass global_feat to Triplet Loss
    # and probabilities to CategoricalCrossentropy
    model = models.Model(inputs=inputs, outputs=[global_feat, probabilities])

    # 2. Define the ID (Classification) Loss
    # Assuming your model outputs are named 'gem_features' and 'id_loss_output'
    id_loss_fn = tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.1, 
        name='id_loss'
    )
    
    # 3. Compile the Model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3.5e-4),
        loss={
            'gem_features': TripletSemiHardLoss(margin=0.3),
            'id_loss_output': id_loss_fn
        },
        loss_weights={'gem_features': 5.0, 'id_loss_output': 1.0},
        metrics={'id_loss_output': 'accuracy'}
    )
    
    return model 
    
train_data, val_data, test_data, prep_layer = get_keras_dataset()
model = build_mobilenet_v1_bnneck_model(prep_layer)
model.summary()

# Callbacks
# 1. Custom Warmup Callback (Matches Paper's Linear Growth)
def lr_schedule_bottleneck(epoch):
    target_lr = 3.5e-4
    warmup_epochs = 10
    
    # 1. Linear Warmup (Epoch 0-9)
    if epoch < warmup_epochs:
        initial_lr = target_lr / 10
        return initial_lr + (target_lr - initial_lr) * (epoch / warmup_epochs)
    
    # 2. MultiStep Decay Milestones
    # Adjusted for EMNIST's faster convergence
    if epoch < 25:
        return target_lr
    elif epoch < 40:
        return target_lr * 0.1  # First decay (10%)
    else:
        return target_lr * 0.01 # Second decay (1%)

class ThroughputPerEpoch(callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()
    def on_epoch_end(self, epoch, logs=None):
        global BATCH_SIZE
        duration = time.time() - self.epoch_start
        throughput = len(train_data) * BATCH_SIZE / duration
        print(f" - Epoch {epoch+1} Time: {duration:.2f}s | Throughput: {throughput:.2f} img/sec")

class ShiftConsistencyCallback(tf.keras.callbacks.Callback):
    def __init__(self, validation_data):
        super().__init__()
        self.validation_data = validation_data

    def on_epoch_end(self, epoch, logs=None):
        total_matches = 0
        total_samples = 0

        for images, _ in self.validation_data:
            # 1. Get predictions for original images
            # Model now returns two outputs: [global_feat, probabilities]
            _, preds_orig = self.model.predict(images, verbose=0)
            labels_orig = np.argmax(preds_orig, axis=1)

            # 2. Create shifted images (Shift 1 pixel right/horizontally)
            # axis=2 corresponds to the width dimension in (batch, height, width, channels)
            images_shifted = tf.roll(images, shift=1, axis=2)

            # 3. Get predictions for shifted images
            _, preds_shifted = self.model.predict(images_shifted, verbose=0)
            labels_shifted = np.argmax(preds_shifted, axis=1)

            # 4. Compare and aggregate
            total_matches += np.sum(labels_orig == labels_shifted)
            total_samples += tf.shape(images)[0].numpy()

        consistency = total_matches / total_samples
        print(f" - Epoch {epoch+1} Shift Consistency (1px): {consistency * 100:.2f}%")

        # Optionally add it to logs so it can be viewed in history or TensorBoard
        if logs is not None:
            logs['val_shift_consistency'] = consistency

# Option 1: Total Loss
# early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=6,
#                                      restore_best_weights=True, verbose=1)
# Option 2:Monitor the validation accuracy of the ID/Classification branch
early_stop = callbacks.EarlyStopping(patience=6, restore_best_weights=True,
                                     mode='max', # Use 'max' for accuracy
                                     verbose=1,
                                     monitor='val_id_loss_output_accuracy',
)
checkpoint = callbacks.ModelCheckpoint("emnist_best_model.keras",
                                       monitor='val_id_loss_output_accuracy',
                                       save_best_only=True, mode='max', verbose=1)
# Wrap it in a Keras Callback
reduce_lr_callback = tf.keras.callbacks.LearningRateScheduler(lr_schedule_bottleneck, 
                                                              verbose=1)

time_callback = ThroughputPerEpoch()
# Shift Consistency callback
shift_callback = ShiftConsistencyCallback(val_data)

def map_dual_labels(image, label_one_hot):
    # 'label_one_hot' is now a vector like [0, 1, 0...] because of label_mode='categorical'
    # The Triplet Loss (gem_features) still needs the integer for mining.
    # We convert it back to an integer ONLY for that specific output.
    label_int = tf.argmax(label_one_hot, axis=-1)
    
    return image, {
        'gem_features': label_int,        # Triplet branch needs integer
        'id_loss_output': label_one_hot   # ID branch needs one-hot for smoothing 
    }

# Apply this to your datasets
train_data = train_data.map(map_dual_labels)
val_data = val_data.map(map_dual_labels)
test_data = test_data.map(map_dual_labels)

# Train
# Record the start time
start_time = time.perf_counter() #

# Train
history = model.fit(
    train_data,
    epochs=200,
    validation_data=val_data,
    callbacks=[early_stop, checkpoint, reduce_lr_callback, time_callback, shift_callback],
    verbose=1
)

# Record the end time
end_time = time.perf_counter() #

# 1. Access the GeM layer from your trained model
# Assuming the layer is named 'ge_m_pooling2d'
gem_layer = model.get_layer('raw_gem_embedding_layer') # Changed to 'gem_features' for BNNeck model
# This weight array has the shape (1, 1, 1, 64).
# Calling .flatten() converts that 4D array into a simple 1D array of 128 values,
# which is exactly what your bar chart needs for plotting
p_values = gem_layer.get_weights()[0].flatten()

# Plot Evaluation Curves

In [ ]:
# ===============================================================
#  STEP 8: Plot Accuracy , Loss, And p-Values (GeM),
# and Confusion Matrix, ROC-Curve, PR-Curve
# ==============================================================
def plot_evaluation_curves(y_test_true, y_test_probs, y_test_pred, 
                           target_names, set_name="Test"):
  # Plotting
  plt.figure(figsize=(12,4))
  plt.subplot(1,2,1)
  plt.plot(history.history['id_loss_output_accuracy'], label='Train Acc')
  plt.plot(history.history['val_id_loss_output_accuracy'], label='Val Acc')
  plt.title('Accuracy over Epochs')
  plt.legend()

  plt.subplot(1,2,2)
  # 1. Plot Classification (ID) Loss
  plt.plot(history.history['id_loss_output_loss'], label='Train ID Loss')
  plt.plot(history.history['val_id_loss_output_loss'], label='Val ID Loss')
  # 2. Plot Triplet Loss
  plt.plot(history.history['gem_features_loss'], label='Train Triplet Loss')
  plt.plot(history.history['val_gem_features_loss'], label='Val Triplet Loss')
  # 3. Plot Combined Total Loss
  plt.plot(history.history['loss'], label='Total Combined Loss', linestyle='--')
  plt.title('Loss Components over Epochs')
  plt.ylabel('Loss Value')
  plt.xlabel('Epoch')
  plt.legend()
  plt.tight_layout()
  plt.show()

  # Plot learning rate Luo et al. (2019) 
  # Simulate 50 epochs
  epochs = np.arange(0, 50)
  lr_values = [lr_schedule_bottleneck(e) for e in epochs]

  plt.figure(figsize=(10, 5))
  plt.plot(epochs, lr_values, label='Learning Rate', color='darkorange', marker='o', markersize=4)
  plt.axvline(x=10, color='red', linestyle='--', label='Warmup End (BN stabilized)')
  plt.axvline(x=25, color='green', linestyle='--', label='1st Decay (Fine-tuning)')
  plt.axvline(x=40, color='blue', linestyle='--', label='2nd Decay (Convergence)')

  plt.title("Luo et al. (2019) 'Strong Baseline' LR Schedule for EMNIST")
  plt.xlabel("Epoch")
  plt.ylabel("Learning Rate")
  plt.yscale('log') # Log scale helps see the small values at the end
  plt.grid(True, which="both", ls="-", alpha=0.5)
  plt.legend()
  plt.show()

  # 2. Plot the distribution of p across the 64 channels
  plt.figure(figsize=(10,8))
  plt.bar(range(len(p_values)), p_values, color='skyblue')
  plt.axhline(y=1.0, color='r', linestyle='--', label='Average Pooling (p=1)')
  plt.axhline(y=3.0, color='g', linestyle='--', label='Starting p=3')
  plt.title("Learned p-values per Channel")
  plt.xlabel("Channel Index")
  plt.ylabel("p value")
  plt.legend()
  plt.show()

  plt.figure(figsize=(10,8))
  # Confusion Matrix
  sns.heatmap(confusion_matrix(y_test_true, y_test_pred),
              annot=True,
              fmt='d',
              cmap='Blues',
              xticklabels=target_names,
              yticklabels=target_names)
  plt.title('Confusion Matrix (Test Set)')
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')
  plt.show()

  # ROC Curve and Precision-Recall Curve
  n_classes = len(target_names)
  y_true_bin = label_binarize(y_test_true, classes=range(n_classes))

  plt.figure(figsize=(12, 6))
  plt.subplot(1, 2, 1)
  for i in range(n_classes):
      fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_test_probs[:, i])
      plt.plot(fpr, tpr, label=f'Class {target_names[i]} (AUC = {auc(fpr, tpr):.3f})')
  plt.plot([0, 1], [0, 1], 'k--')
  plt.title(f'ROC Curves - {set_name}')
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

  plt.subplot(1, 2, 2)
  for i in range(n_classes):
      precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_test_probs[:, i])
      plt.plot(recall, precision, label=f'Class {target_names[i]}')
  plt.title(f'PR Curves - {set_name}')
  plt.tight_layout()
  plt.show()


# Evaluation logic
def run_full_evaluation(model, dataset, history, target_names):
    print("Generating predictions for evaluation...")
    
    y_true = []
    y_probs = []
    for images, labels_dict in dataset:

        # labels_dict is {'gem_features': label, 'id_loss_output': label}
        # Extract true labels: Since they are one-hot, use argmax to get the integer
        # Based on your updated map_dual_labels
        true_batch = np.argmax(labels_dict['id_loss_output'].numpy(), axis=-1)
        y_true.extend(true_batch)
        
        # Model returns [gem_features, id_loss_output]
        _, probs = model.predict(images, verbose=0)
        y_probs.extend(probs)

    y_test_true = np.array(y_true)
    y_test_probs = np.array(y_probs)
    y_test_pred = np.argmax(y_probs, axis=-1)

    # Print classification report
    print("Classification Report (Test Set):")
    print(classification_report(y_test_true, y_test_pred, target_names=target_names))

    # 2. Print accuracy on test set
    # The order is: [total_loss, gem_features_loss, id_loss_output_loss, id_loss_output_accuracy]
    results = model.evaluate(test_data, verbose=2)
    total_loss = results[0]
    gem_loss = results[1]
    id_loss = results[2]
    test_acc = results[3]

    print(f"\nTest Accuracy: {test_acc:.4f}")
    print(f"Total Test Loss: {total_loss:.4f} \n gem_loss: {gem_loss:.4f} \n id_loss: {id_loss:.4f}")
 
    # 3. Plot Evaluation curve for test set
    print("Plotting evaluation curves...")
    plot_evaluation_curves(y_test_true, y_test_probs, 
                           y_test_pred, target_names, set_name="Test")
    
target_names = [str(i) for i in range(10)]
run_full_evaluation(model, test_data, history, target_names)


# Measure Latency metric

In [ ]:
# measure precise latency
def measure_precise_latency(model, input_shape=(28, 28, 1), num_samples=100):
    """
    Measures the inference latency of a single model.predict() call.
    Environment: Optimized for Kaggle GPU (T4/P100).
    """
    # 1. Create a single dummy input tensor
    dummy_input = np.random.rand(1, *input_shape).astype(np.float32)

    print(f"\n--- Starting Latency Measurement ({num_samples} iterations) ---")

    # 2. WARM-UP: Essential for GPU initialization
    for _ in range(10):
        _ = model.predict(dummy_input, verbose=0)

    # 3. MEASUREMENT LOOP: Using high-precision perf_counter
    latencies = []
    for _ in range(num_samples):
        start_time = time.perf_counter()
        # The model now returns two outputs, we only need the probabilities for latency
        _, _ = model.predict(dummy_input, verbose=0)
        end_time = time.perf_counter()
        latencies.append(end_time - start_time)

    # 4. CALCULATE STATS
    avg_latency_ms = (sum(latencies) / num_samples) * 1000
    p95_latency_ms = np.percentile(latencies, 95) * 1000

    print(f"Average Inference Latency: {avg_latency_ms:.2f} ms")
    print(f"P95 Latency (Tail Latency): {p95_latency_ms:.2f} ms")
    return avg_latency_ms

# 1. Execute the improved precise latency measurement
measure_precise_latency(model, input_shape=(28, 28, 1))


# Checking if TripletSemiHardLoss is successfully restoring its state.

In [ ]:
try:
    triplet_loss_instance = model.loss['gem_features']
    
    # Retrieve the margin from the restored object
    stored_margin = triplet_loss_instance.margin
    
    print(f"--- Verification Success ---")
    print(f"Stored Margin value: {stored_margin}")
    
    # Also check the internal Keras config dictionary
    config = triplet_loss_instance.get_config()
    print(f"Serialized Config: {config}")
    
    if stored_margin == 0.3:
        print("✅ The margin was stored and restored correctly.")
    else:
        print("❌ Margin mismatch detected.")

except KeyError:
    print("Error: Could not find 'gem_features' in the model's loss dictionary.")

# Verifying the "Neck" Statistics
Since you've successfully loaded the model, you can verify if the BNNeck is doing its job `(centering the features for the classifier` while `leaving them uncentered for the triplet loss)` by checking the weights of the bn_neck layer:

In [ ]:
# Access the BatchNormalization 'neck' layer
bn_layer = model.get_layer('bn_neck')

# Check the learned moving mean and moving variance
# Note the correct attribute name: moving_variance
print(f"BN Neck Mean (first 5 channels): {bn_layer.moving_mean.numpy()[:5]}")
print(f"BN Neck Variance (first 5 channels): {bn_layer.moving_variance.numpy()[:5]}")

# TSNE or PCA visualization of the gem_features
This will allow you to see if the `Triplet Loss` is successfully creating distinct clusters for each EMNIST digit in the embedding space.

# What to look for in the plot
## Tight Clusters:
Indicates high intra-class similarity.

## Wide Gaps:
Indicates high inter-class variance (the result of the Triplet Loss).

## Overlapping Points:
If digits like '4' and '9' overlap, it suggests the model needs more training
or a higher margin to distinguish those specific features.


In [ ]:
def plot_tsne_embeddings(model, dataset, num_batches=10):
    embeddings = []
    labels = []

    for i, (images, labs_dict) in enumerate(dataset):
        if i >= num_batches: break

        # model.predict returns a list: [gem_features, id_loss_output]
        preds = model.predict(images, verbose=0)

        # CHANGE: Use index 0 for gem_features (L2-normalized embeddings)
        # Index 1: id_loss_output (The Softmax branch after the BNNeck)
        emb = preds[0]

        embeddings.append(emb)
        # Extract the integer label from the dictionary by taking argmax of the one-hot encoded label
        labels.append(np.argmax(labs_dict['id_loss_output'].numpy(), axis=-1))

    embeddings = np.vstack(embeddings)
    labels = np.concatenate(labels)

    # ... (rest of the TSNE logic remains the same)
    tsne = TSNE(n_components=2, random_state=42)
    embeddings_2d = tsne.fit_transform(embeddings)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                         c=labels, cmap='tab10', alpha=0.6)
    plt.colorbar(scatter, ticks=range(10))
    plt.title("TSNE Visualization of BNNeck GeM Embeddings")
    plt.show()

# Run the visualization
plot_tsne_embeddings(model, test_data)

# How to find the "Hardest Positives" and "Hardest Negatives" from test set to see exactly which images the model is still struggling with?

To identify the images your model struggles with, we need to find the Hardest Positive (the image of the same class furthest away in embedding space) and the Hardest Negative (the image of a different class closest in embedding space). This process directly evaluates the effectiveness of the $0.3$ margin you verified in your loss configuration.

## The Margin Goal: 
Your Triplet Loss is trying to make sure that the `Hard Positive Dist + 0.3 < Hard Negative Dist`. If the distances are very close, it indicates the model needs more training epochs.

## The "Neck" Role: 
If your `classification accuracy is high but the triplet distances look messy, it means the BNNeck is doing its job—allowing` the classifier to work on centered data while the `triplet branch struggles to organize the raw metric space`. `If both are good, your backbone has successfully learned a "Strong Baseline" representation`.

# How to Interpret the Results
## Hard Positive Distance > 0.3: 
If the distance to the "Hard Pos" is greater than your margin ($0.3$), the m`odel is failing to group those similar images together`. This often happens with digits that have unusual stroke widths or rotations.

## Hard Negative Distance < 0.3: 
If a "Hard Neg" is very close to the anchor (low distance), the model is `confusing two different classes`. Look for similarities between '4' and '9' or '5' and '6'.

## Success Indicator: 
### A "healthy" model should show a Hard Positive distance that is significantly smaller than the Hard Negative distance.

# Why this Validates your BNNeckThis?
visualization looks at the Metric Branch of your architecture (before the BNNeck).
By seeing exactly which images are close in the embedding space, you are verifying
if the `Backbone → GeM → L2Normalize` flow is creating a discriminative space where `identity (digit class)` is the primary feature, which is the core goal of the BNNeck design.

# How to Improve These Results
If you want those "Hard Negatives" to move further away, consider these steps:

## Increase Training Epochs: 
`Triplet loss usually requires significantly more time to converge than standard Cross-Entropy`. Since you only ran for a few epochs, the "clustering" is just beginning.

## Adjust the Margin: 
`If the Hard Negatives stay consistently around 0.3, you might increase the margin to 0.5 to force the model to work harder on separation`.

## Warm-up Period: 
`Some BNNeck implementations freeze the ID branch for a few epochs to let the Triplet branch "organize" the feature space first`.


In [ ]:
def visualize_hard_triplets_final(model, dataset, num_anchors=5):
    found_diverse_batch = False

    # 1. More aggressive shuffling to break class-ordering in folders
    # We unbatch, shuffle with a large buffer, and take a large batch (128)
    print("Searching for a diverse batch of images...")

    # Try up to 10 times to find a batch with multiple classes
    for attempt in range(10):
        diverse_sample = dataset.unbatch().shuffle(10000).batch(128).take(1)

        for images, labels_dict in diverse_sample:
            
            labels = np.argmax(labels_dict['id_loss_output'].numpy(), axis=-1)
            unique_classes = np.unique(labels)

            if len(unique_classes) < 2:
                continue

            found_diverse_batch = True
            print(f"✅ Success! Found {len(unique_classes)} unique classes in the batch.")

            # 2. Extract Embeddings (Index 0 is gem_features/L2 branch)
            preds = model.predict(images, verbose=0)
            embeddings = preds[0]

            # 3. Calculate Euclidean Distance Matrix
            dot_product = np.dot(embeddings, embeddings.T)
            square_norm = np.diag(dot_product)
            distances = np.maximum(square_norm[:, None] - 2.0 * dot_product + square_norm[None, :], 0.0)
            distances = np.sqrt(distances)

            plt.figure(figsize=(15, num_anchors * 4))

            found_count = 0
            for i in range(len(labels)):
                if found_count >= num_anchors: break

                anchor_label = labels[i]
                mask_pos = (labels == anchor_label)
                mask_neg = (labels != anchor_label)

                # Verify we have at least one other image of the same class
                # and at least one image of a different class
                if np.sum(mask_pos) < 2 or np.sum(mask_neg) == 0:
                    continue

                # Hardest Positive: Max distance to same-class image (ignore self)
                pos_indices = np.where(mask_pos)[0]
                pos_dists = distances[i][mask_pos]
                pos_dists[np.where(pos_indices == i)[0]] = -1 # Ignore self
                hard_pos_idx = pos_indices[np.argmax(pos_dists)]

                # Hardest Negative: Min distance to different-class image
                neg_indices = np.where(mask_neg)[0]
                neg_dists = distances[i][mask_neg]
                hard_neg_idx = neg_indices[np.argmin(neg_dists)]

                # 4. Plotting
                triplet_idx = [i, hard_pos_idx, hard_neg_idx]
                titles = ['Anchor', 'Hard Positive', 'Hard Negative']

                for col, idx in enumerate(triplet_idx):
                    plt.subplot(num_anchors, 3, found_count * 3 + col + 1)
                    plt.imshow(images[idx].numpy().squeeze(), cmap='gray')
                    plt.title(f"{titles[col]} (Class {labels[idx]})\nDist: {distances[i, idx]:.3f}")
                    plt.axis('off')

                found_count += 1

            plt.tight_layout()
            plt.show()
            return # Exit after one successful plot

    print("❌ Could not find a diverse batch after 10 attempts. Check if your test_data is correctly loaded.")

# Run the visualization
visualize_hard_triplets_final(model, test_data)

# Create a Confusion Matrix to see if these "Hard Negatives" are causing specific misclassifications in your classification branch?
To evaluate if the BNNeck (the Batch Normalization layer before the final classification head) is successfully "fixing" the overlap we saw in the metric branch, we can generate a Confusion Matrix.

This will show us if the digits that were "Hard Negatives" in the triplet branch (like a '1' being close to a '7') are actually causing misclassifications in the final output.

## Classification Performance (ID Branch) Analysis
Run this code to generate and plot the confusion matrix for your classification output:

## How to interpret this vs. your "Hard Negatives"?
### The "Correction" Effect: 
Compare the errors in this matrix to the result.png you uploaded. In your triplet visualization, you saw that a '7' (Anchor) 
was very close to a '1' (Hard Negative).

1. If the Confusion Matrix shows few errors for 7s: It means the BNNeck is working perfectly. 
Even though the metric space is crowded, the normalization and softmax head successfully separated them for the final prediction.

2. If the Confusion Matrix shows many 7s predicted as 1s: It means the overlap in the metric space is too severe for even the BNNeck to 
fix.

### Diagonal Strength: 
A strong green diagonal indicates that your model’s ID Loss training was successful. Since your previous logs 
showed $\approx 73\%$ accuracy, you should see a mostly clear diagonal with some specific off-diagonal "hotspots.

### "Specific Confusion Pairs: 
Look for the highest numbers outside the diagonal. Common EMNIST confusion pairs are (4, 9), (5, S/6), and (1, 7). 
These "hotspots" will correspond exactly to the digits that appeared most frequently as Hard Negatives in your previous visualization.

## The Power of BNNeck
The magic of the BNNeck architecture is that it allows the model to satisfy two conflicting goals at once:

### Triplet Branch: 
Learns to put similar things together on a hypersphere (Metric learning).
### ID Branch: 
Learns to pull things across a decision boundary (Classification).

By comparing these two results, you can decide if your model needs a higher margin (to help the Triplet branch) or more regularization 
(to help the Classification branch).


In [ ]:
def plot_id_branch_confusion_matrix(model, dataset):
    y_true = []
    y_pred = []

    print("Gathering classification results...")
    for images, labels_dict in dataset:
        # 1. Extract true labels from the dictionary
        # Correctly get integer labels for the batch
        labels = np.argmax(labels_dict['id_loss_output'].numpy(), axis=-1)
        y_true.extend(labels.tolist()) # Convert to list to extend

        # 2. Get predictions (Index 1 is the 'id_loss_output' softmax branch)
        preds = model.predict(images, verbose=0)
        class_probs = preds[1]

        # 3. Get the predicted digit (highest probability)
        predicted_classes = np.argmax(class_probs, axis=1)
        y_pred.extend(predicted_classes.tolist())

    # Convert to numpy for sklearn
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Compute the Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plotting
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=range(10), yticklabels=range(10))
    plt.xlabel('Predicted Digit (Classification Branch)')
    plt.ylabel('True Digit (Label)')
    plt.title('Confusion Matrix: BNNeck Classification Accuracy')
    plt.show()

# Execute the plot
plot_id_branch_confusion_matrix(model, test_data)

# Out Of Sample Testing


In [ ]:
# 1. Path to the Synthetic Dataset
DATA_DIR = "/kaggle/input/datasets/osamaaslam86004/greyscale-synthetic-font-digits-blanaced/greyscale_synthetic_digits"

# Constants
IMG_SIZE = (28, 28)
BATCH_SIZE = 128

def get_oos_dataset():
    print("--- Loading Synthetic Font Digits ---")

    # 2. Load the dataset (28x28, Grayscale, 10 Classes)
    oos_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        labels='inferred',
        label_mode='int',
        color_mode='grayscale',
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False  # Crucial for label comparison
    )

    # 4. Normalization layer (NO augmentation)
    norm_layer = layers.Normalization(axis=None)
    # Adapt on a mapping to ensure the stats are based on [0, 1] range
    norm_layer.adapt(oos_ds.map(lambda x, y: x / 255.0).take(500))

    # Note: We do NOT use RandomRotation/Zoom/Translation for testing.
    # Testing should only include Rescaling and Normalization to match training.
    # Assuming 'prep_layer' from your training is already adapted:
    preprocessing_model = models.Sequential([
        layers.Rescaling(1./255),
        norm_layer
    ], name="inference_preprocessing")

    return oos_ds, preprocessing_model

# Get the dataset
oos_ds_raw, inference_prep = get_oos_dataset()


# ===============================================================
#  STEP 8: Plot Accuracy , Loss, And p-Values (GeM),
# and Confusion Matrix, ROC-Curve, PR-Curve
# ==============================================================
print("Running inference on Out-of-Sample synthetic digits...")
def plot_evaluation_curves(y_test_true, y_test_probs, y_test_pred, 
                           target_names, set_name="Test"):

  # Plot the distribution of p across the 64 channels
  plt.figure(figsize=(10,8))
  plt.bar(range(len(p_values)), p_values, color='skyblue')
  plt.axhline(y=1.0, color='r', linestyle='--', label='Average Pooling (p=1)')
  plt.axhline(y=3.0, color='g', linestyle='--', label='Starting p=3')
  plt.title("Learned p-values per Channel")
  plt.xlabel("Channel Index")
  plt.ylabel("p value")
  plt.legend()
  plt.show()

  # Confusion Matrix
  plt.figure(figsize=(10,8))
  sns.heatmap(confusion_matrix(y_test_true, y_test_pred),
              annot=True,
              fmt='d',
              cmap='Blues',
              xticklabels=target_names,
              yticklabels=target_names)
  plt.title('Confusion Matrix (Test Set)')
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')
  plt.show()

  # ROC Curve and Precision-Recall Curve
  n_classes = len(target_names)
  y_true_bin = label_binarize(y_test_true, classes=range(n_classes))

  plt.figure(figsize=(12, 6))
  plt.subplot(1, 2, 1)
  for i in range(n_classes):
      fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_test_probs[:, i])
      plt.plot(fpr, tpr, label=f'Class {target_names[i]} (AUC = {auc(fpr, tpr):.3f})')
  plt.plot([0, 1], [0, 1], 'k--')
  plt.title(f'ROC Curves - {set_name}')
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

  plt.subplot(1, 2, 2)
  for i in range(n_classes):
      precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_test_probs[:, i])
      plt.plot(recall, precision, label=f'Class {target_names[i]}')
  plt.title(f'PR Curves - {set_name}')
  plt.tight_layout()
  plt.show()


def map_dual_labels(image, label_int_input):
    # For oos_ds_raw, label_mode='int', so the label is already an integer tensor.
    # label_int_input is a batch of integer labels, e.g., [7, 2, 0, 9, ...]
    # For id_loss_output (CategoricalCrossentropy), it needs to be one-hot encoded
    num_classes = 10 # EMNIST digits are 0-9
    label_one_hot_output = tf.one_hot(label_int_input, depth=num_classes)

    return image, {
        'gem_features': label_int_input,        # Triplet branch needs integer
        'id_loss_output': label_one_hot_output   # ID branch needs one-hot for smoothing
    }

# Apply this to your datasets
oos_ds = oos_ds_raw.map(map_dual_labels)

# Evaluation logic
def run_full_evaluation(model, dataset, target_names):
    print("Generating predictions for evaluation...")
    
    y_true = []
    y_probs = []
    for images, labels_dict in dataset:

        # labels_dict is {'gem_features': label, 'id_loss_output': label}
        # Extract true labels: Since they are one-hot, use argmax to get the integer
        # Based on your updated map_dual_labels
        true_batch = np.argmax(labels_dict['id_loss_output'].numpy(), axis=-1)
        y_true.extend(true_batch.tolist()) # Convert to list to extend
        
        # Model returns [gem_features, id_loss_output]
        _, probs = model.predict(images, verbose=0)
        y_probs.extend(probs)

    y_test_true = np.array(y_true)
    y_test_probs = np.array(y_probs)
    y_test_pred = np.argmax(y_probs, axis=-1)

    # Print classification report
    print("Classification Report (Test Set):")
    print(classification_report(y_test_true, y_test_pred, target_names=target_names, zero_division=0))

    # 2. Print accuracy on test set
    # The order is: [total_loss, gem_features_loss, id_loss_output_loss, id_loss_output_accuracy]
    results = model.evaluate(test_data, verbose=2)
    total_loss = results[0]
    gem_loss = results[1]
    id_loss = results[2]
    test_acc = results[3]

    print(f"\nTest Accuracy: {test_acc:.4f}")
    print(f"Total Test Loss: {total_loss:.4f} \n gem_loss: {gem_loss:.4f} \n id_loss: {id_loss:.4f}")
 
    # 3. Plot Evaluation curve for test set
    print("Plotting evaluation curves...")
    plot_evaluation_curves(y_test_true, y_test_probs, 
                           y_test_pred, target_names, set_name="Test")
    
target_names = [str(i) for i in range(10)]
run_full_evaluation(model, oos_ds, target_names)